In [ ]:
#!/usr/bin/env python3
# -*- coding: UTF-8 -*-

from os import close
import backtrader as bt
from utility.ToolKit import ToolKit
from datetime import datetime
from datetime import datetime, timedelta
from backtrader_plotting import Bokeh
from backtrader_plotting.schemes import Tradimo
from backtraderref.BTStrategyVol import BTStrategyVol
from utility.TickerInfo import TickerInfo
from backtraderref.BTPandasDataExt import BTPandasDataExt
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pyfolio as pf
import pandas as pd


if __name__ == "__main__":

    """创建cerebro对象"""
    cerebro = bt.Cerebro()
    """ 添加bt相关的策略 """
    cerebro.addstrategy(BTStrategyVol)
    """ 初始资金100M """
    cerebro.broker.setcash(1000000.0)
    """ 每手10股 """
    cerebro.addsizer(bt.sizers.FixedSize, stake=100)
    """ 费率千分之一 """
    cerebro.broker.setcommission(commission=0.001, stocklike=True)
    """ 添加股票当日即历史数据 """
    trade_date = ToolKit("获取最新A股交易日期").get_cn_latest_trade_date(1)
    list = TickerInfo(trade_date, "cn").get_backtrader_data_feed()
    """ 循环初始化数据进入cerebro """
    for h in list:
        print("正在初始化: ", h["symbol"][0])
        """ 历史数据最早不超过2021-01-01 """
        data = BTPandasDataExt(
            dataname=h, name=h["symbol"][0], fromdate=datetime(2021, 1, 1)
        )
        cerebro.adddata(data)

    print("Starting Portfolio Value: %.2f" % cerebro.broker.getvalue())

    # 回测时需要添加 TimeReturn 分析器
    cerebro.addanalyzer(bt.analyzers.TimeReturn, _name='_TimeReturn')
    result = cerebro.run()

    print("当前现金持有: ", cerebro.broker.get_cash())
    print("Final Portfolio Value: %.2f" % cerebro.broker.getvalue())

    # 提取收益序列
    pnl = pd.Series(result[0].analyzers._TimeReturn.get_analysis())

    # 计算累计收益
    cumulative = (pnl + 1).cumprod()

    # 计算回撤序列
    max_return = cumulative.cummax()

    drawdown = (cumulative - max_return) / max_return

    # 按年统计收益指标
    perf_stats_year = (
        (pnl)
        .groupby(pnl.index.to_period("y"))
        .apply(lambda data: pf.timeseries.perf_stats(data))
        .unstack()
    )

    # 统计所有时间段的收益指标
    perf_stats_all = pf.timeseries.perf_stats((pnl)).to_frame(name="all")

    perf_stats = pd.concat([perf_stats_year, perf_stats_all.T], axis=0)

    perf_stats_ = round(perf_stats, 4).reset_index()


    # 绘制图形

    plt.rcParams["axes.unicode_minus"] = False  # 用来正常显示负号

    # 导入设置坐标轴的模块
    plt.style.use("seaborn")
    plt.style.use('dark_background')

    fig, (ax0, ax1) = plt.subplots(
        2, 1, gridspec_kw={"height_ratios": [1.5, 4]}, figsize=(20, 8)
    )

    cols_names = [
        "date",
        "Annual\nreturn",
        "Cumulative\nreturns",
        "Annual\nvolatility",
        "Sharpe\nratio",
        "Calmar\nratio",
        "Stability",
        "Max\ndrawdown",
        "Omega\nratio",
        "Sortino\nratio",
        "Skew",
        "Kurtosis",
        "Tail\nratio",
        "Daily value\nat risk",
    ]


    # 绘制表格
    ax0.set_axis_off()
    # 除去坐标轴
    table = ax0.table(
        cellText=perf_stats_.values,
        bbox=(0, 0, 1, 1),
        # 设置表格位置， (x0, y0, width, height)
        rowLoc="right",
        # 行标题居中
        cellLoc="right",
        colLabels=cols_names,
        # 设置列标题
        colLoc="right",
        # 列标题居中
        edges="open",  # 不显示表格边框
    )

    table.set_fontsize(13)


    # 绘制累计收益曲线
    ax2 = ax1.twinx()

    ax1.yaxis.set_ticks_position("right")
    # 将回撤曲线的 y 轴移至右侧
    ax2.yaxis.set_ticks_position("left")
    # 将累计收益曲线的 y 轴移至左侧
    # 绘制回撤曲线
    drawdown.plot.area(
        ax=ax1, label="drawdown (right)", rot=0, alpha=0.3, fontsize=13, grid=False
    )

    # 绘制累计收益曲线
    (cumulative).plot(
        ax=ax2,
        color="#F1C40F",
        lw=3.0,
        label="cumret (left)",
        rot=0,
        fontsize=13,
        grid=False,
    )

    # 不然 x 轴留有空白
    ax2.set_xbound(lower=cumulative.index.min(), upper=cumulative.index.max())

    # 主轴定位器：每 5 个月显示一个日期：根据具体天数来做排版
    ax2.xaxis.set_major_locator(ticker.MultipleLocator(100))

    # 同时绘制双轴的图例
    h1, l1 = ax1.get_legend_handles_labels()

    h2, l2 = ax2.get_legend_handles_labels()

    plt.legend(h1 + h2, l1 + l2, fontsize=12, loc="upper left", ncol=1)


    fig.tight_layout()
    # 规整排版
    plt.show()

Loading BokehJS ...

/root/miniconda3/lib/python3.9/site-packages/pyfolio/pos.py:26: UserWarning: Module "zipline.assets" not found; mutltipliers will not be applied to position notionals.
  warnings.warn(



获取最新A股交易日期...
当前获取的A股日期是： 20220223

获取上一个交易日...
当前获取的A股日期是： 20220223

加载历史数据...
正在处理股票：正在处理股票：正在处理股票：正在处理股票：正在处理股票：  SZ300086 正在处理股票：正在处理股票：正在处理股票： SZ300249
  SZ301130 
 SZ300605正在处理股票：
SZ300338
 
正在处理股票：SZ300782SZ300666SZ300199SZ300584 正在处理股票：

正在处理股票：
正在处理股票：正在处理股票：
正在处理股票： SZ300387正在处理股票： 
正在处理股票：SZ300604   SZ002325 
SZ000595SZ300665
 
SZ300655SZ300343
正在处理股票：正在处理股票：

SZ300846 
正在处理股票：正在处理股票： 正在处理股票： 正在处理股票： SZ000859
SZ002354 SZ002381

SZ000788
SZ000816正在处理股票：正在处理股票：正在处理股票：正在处理股票：正在处理股票： 
  正在处理股票： 正在处理股票： SZ002873 SZ002326
SZ002308

  SZ002621SZ002617
SZ002125正在处理股票：
正在处理股票：SZ002497
 正在处理股票： 正在处理股票：SZ002402 SZ003042 
正在处理股票：SZ002644SZ002748


SZ000987 

SZ002955正在处理股票：正在处理股票：正在处理股票：
 正在处理股票：SZ002162正在处理股票：正在处理股票：正在处理股票：正在处理股票：   SZ002999 

SZ002667SZ002132

   
正在处理股票：SZ002771SZ002725正在处理股票：
SZ002842
 SZ300373SZ002068正在处理股票：
 
 正在处理股票：正在处理股票：正在处理股票：正在处理股票：SZ000532  
SZ300167SZ300035

正在处理股票：正在处理股票： SZ000766正在处理股票：正在处理股票： 正在处理股票：SZ300346 

SZ002484
正在处理股票： SZ300821 正在处理股票：
SZ300555

  正在处理股票：SZ002248SZ002010 

SZ002357正在处理股票：
正在处理股票：

  SZ000756  SZ000712
SZ002243SZ002939
SZ000012正在处理股票：
正在处理股票：正在处理股票：正在处理股票：正在处理股票：正在处理股票： 正在处理股票：正在处理股票：  SZ002803 SZ002046SZ300003
    SZ300068
正在处理股票：SZ000999
SZ000039SZ000851SZ000823
正在处理股票： 

SZ300520正在处理股票：

正在处理股票：正在处理股票：
正在处理股票：正在处理股票：   SZ002445SZ000892  SZ300419
SZ300085SZ002417


 正在处理股票：正在处理股票： SZ000858SZ002701

 SZ300099

正在处理股票：正在处理股票：正在处理股票：  SZ002317SZ300181
 正在处理股票：SZ300015正在处理股票：
正在处理股票：
 正在处理股票：正在处理股票： SZ300471 SZ002433正在处理股票：正在处理股票：正在处理股票： SZ002658

  SZ300272正在处理股票：SZ300748
   
SZ300198正在处理股票：正在处理股票：
SZ000533SZ002600SZ002279正在处理股票：
正在处理股票：


 SZ002684 SZ300253
  正在处理股票：
正在处理股票：SZ000545 
正在处理股票：正在处理股票：正在处理股票：SZ000989
SZ300179正在处理股票： 正在处理股票：正在处理股票： 
  SZ002050正在处理股票：SZ002640SZ002762 
SZ300569

   SZ300024
正在处理股票：
 SZ300002
SZ002565SZ301217

正在处理股票：正在处理股票： 正在处理股票：SZ002173正在处理股票：正在处理股票：SZ300297 正在处理股票： 正在处理股票：
SZ000157
 
SZ300040  正在处理股票： SZ002335正在处理股票：SZ002618
正在处理股票： 
SZ002001SZ002893正在处理股票：SZ300352
   正在处理股票： SZ30

正在处理股票：SH600487SH601872正在处理股票： 
 SH601619SH600733

正在处理股票： SH600805
  SH600570正在处理股票：SH600691正在处理股票： SH600372
正在处理股票：正在处理股票：  
SH600552正在处理股票：正在处理股票：SH600963
SH600588
 

正在处理股票：SH600072 SH601901
 正在处理股票：正在处理股票：正在处理股票：SH600299    SH600989SH600276SH600559

正在处理股票：

 SH600716

SH600557
正在处理股票： 正在处理股票：SH600706 SH600257正在处理股票：
 正在处理股票：正在处理股票：正在处理股票：正在处理股票：SH601877   
SH600331SH600031SH603169


正在处理股票： 正在处理股票：
正在处理股票： 正在处理股票：SH600595正在处理股票：  正在处理股票：正在处理股票：  SH600643SH600338  SH600537
SH601798

正在处理股票：SH600516SH600268 

SH600273
正在处理股票：
 SH603088

SH600410
正在处理股票：正在处理股票：正在处理股票：  SH600602SH600623

 正在处理股票：正在处理股票：正在处理股票：正在处理股票：正在处理股票：正在处理股票：    SH600905SH600409正在处理股票：SH600722SH601919 
 正在处理股票：SH600075
正在处理股票：
SH601778
  

SH600105SH600497
 正在处理股票：正在处理股票：SH601218  

正在处理股票：SH600901正在处理股票： SH600586正在处理股票：
SH600104  
SH600186SH601868
SH600111

正在处理股票：
正在处理股票：正在处理股票： 正在处理股票：正在处理股票：正在处理股票：正在处理股票：  SH600906SH600804

 正在处理股票：SH600235   
正在处理股票： SH600021SH601890正在处理股票：

SH600405SH600291SH600690 正在处理股票：

...

2021-01-05, Buy SH600808 Created 2.50
2021-01-05, Buy SH600572 Created 4.61
2021-01-05, Buy SH600307 Created 1.63
2021-01-05, Buy SH600300 Created 5.00
2021-01-05, Buy SH600837 Created 12.63
2021-01-05, Buy SH600479 Created 7.78
2021-01-05, Buy SH600999 Created 22.96
2021-01-05, Buy SH600106 Created 2.62
2021-01-05, Buy SH600155 Created 12.81
2021-01-05, Buy SH601216 Created 4.50
2021-01-05, Buy SH600854 Created 3.72
2021-01-05, Buy SH600010 Created 1.17
2021-01-05, Buy SH601991 Created 2.22
2021-01-05, Buy SH600773 Created 7.32
2021-01-05, Buy SH601117 Created 5.53
2021-01-05, Buy SH601928 Created 5.94
2021-01-05, Buy SH601139 Created 6.95
2021-01-05, Buy SH603000 Created 17.19
2021-01-05, Buy SH600868 Created 3.03
2021-01-05, Buy SH601718 Created 3.05
2021-01-05, Buy SH600535 Created 15.02
2021-01-05, Buy SH600251 Created 7.75
2021-01-05, Buy SH600567 Created 3.05
2021-01-05, Buy SH603887 Created 14.86
2021-01-05, Buy SH600979 Created 3.25
2021-01-05, Buy SH600741 Created 28.08
2021-

2021-01-06, Buy SH600316 Order Canceled/Margin/Rejected
2021-01-06, Buy SH600570 Order Canceled/Margin/Rejected
2021-01-06, Buy SH603306 Order Canceled/Margin/Rejected
2021-01-06, Buy SH600550 Order Canceled/Margin/Rejected
2021-01-06, Buy SH600183 Order Canceled/Margin/Rejected
2021-01-06, Buy SH600500 Order Canceled/Margin/Rejected
2021-01-06, Buy SH600487 Order Canceled/Margin/Rejected
2021-01-06, Buy SH601872 Order Canceled/Margin/Rejected
2021-01-06, Buy SH600691 Order Canceled/Margin/Rejected
2021-01-06, Buy SH600552 Order Canceled/Margin/Rejected
2021-01-06, Buy SH601619 Order Canceled/Margin/Rejected
2021-01-06, Buy SH600733 Order Canceled/Margin/Rejected
2021-01-06, Buy SH600805 Order Canceled/Margin/Rejected
2021-01-06, Buy SH600299 Order Canceled/Margin/Rejected
2021-01-06, Buy SH600372 Order Canceled/Margin/Rejected
2021-01-06, Buy SH600588 Order Canceled/Margin/Rejected
2021-01-06, Buy SH600963 Order Canceled/Margin/Rejected
2021-01-06, Buy SH600072 Order Canceled/Margin/R

2021-01-07, Buy SH600316 Order Canceled/Margin/Rejected
2021-01-07, Buy SH600570 Order Canceled/Margin/Rejected
2021-01-07, Buy SH603306 Order Canceled/Margin/Rejected
2021-01-07, Buy SH600550 Order Canceled/Margin/Rejected
2021-01-07, Buy SH600183 Order Canceled/Margin/Rejected
2021-01-07, Buy SH600500 Order Canceled/Margin/Rejected
2021-01-07, Buy SH600487 Order Canceled/Margin/Rejected
2021-01-07, Buy SH601872 Order Canceled/Margin/Rejected
2021-01-07, Buy SH600691 Order Canceled/Margin/Rejected
2021-01-07, Buy SH600552 Order Canceled/Margin/Rejected
2021-01-07, Buy SH601619 Order Canceled/Margin/Rejected
2021-01-07, Buy SH600733 Order Canceled/Margin/Rejected
2021-01-07, Buy SH600805 Order Canceled/Margin/Rejected
2021-01-07, Buy SH600299 Order Canceled/Margin/Rejected
2021-01-07, Buy SH600372 Order Canceled/Margin/Rejected
2021-01-07, Buy SH600588 Order Canceled/Margin/Rejected
2021-01-07, Buy SH600963 Order Canceled/Margin/Rejected
2021-01-07, Buy SH600072 Order Canceled/Margin/R

2021-01-08, Buy SZ300182 Order Canceled/Margin/Rejected
2021-01-08, Buy SH600316 Order Canceled/Margin/Rejected
2021-01-08, Buy SH600570 Order Canceled/Margin/Rejected
2021-01-08, Buy SH603306 Order Canceled/Margin/Rejected
2021-01-08, Buy SH600550 Order Canceled/Margin/Rejected
2021-01-08, Buy SH600183 Order Canceled/Margin/Rejected
2021-01-08, Buy SH600500 Order Canceled/Margin/Rejected
2021-01-08, Buy SH600487 Order Canceled/Margin/Rejected
2021-01-08, Buy SH601872 Order Canceled/Margin/Rejected
2021-01-08, Buy SH600691 Order Canceled/Margin/Rejected
2021-01-08, Buy SH600552 Order Canceled/Margin/Rejected
2021-01-08, Buy SH601619 Order Canceled/Margin/Rejected
2021-01-08, Buy SH600733 Order Canceled/Margin/Rejected
2021-01-08, Buy SH600805 Order Canceled/Margin/Rejected
2021-01-08, Buy SH600299 Order Canceled/Margin/Rejected
2021-01-08, Buy SH600372 Order Canceled/Margin/Rejected
2021-01-08, Buy SH600588 Order Canceled/Margin/Rejected
2021-01-08, Buy SH600963 Order Canceled/Margin/R

2021-01-11, Buy SZ300182 Order Canceled/Margin/Rejected
2021-01-11, Buy SH600316 Order Canceled/Margin/Rejected
2021-01-11, Buy SH600570 Order Canceled/Margin/Rejected
2021-01-11, Buy SH603306 Order Canceled/Margin/Rejected
2021-01-11, Buy SH600550 Order Canceled/Margin/Rejected
2021-01-11, Buy SH600183 Order Canceled/Margin/Rejected
2021-01-11, Buy SH600500 Order Canceled/Margin/Rejected
2021-01-11, Buy SH600487 Order Canceled/Margin/Rejected
2021-01-11, Buy SH601872 Order Canceled/Margin/Rejected
2021-01-11, Buy SH600691 Order Canceled/Margin/Rejected
2021-01-11, Buy SH600552 Order Canceled/Margin/Rejected
2021-01-11, Buy SH601619 Order Canceled/Margin/Rejected
2021-01-11, Buy SH600733 Order Canceled/Margin/Rejected
2021-01-11, Buy SH600805 Order Canceled/Margin/Rejected
2021-01-11, Buy SH600299 Order Canceled/Margin/Rejected
2021-01-11, Buy SH600372 Order Canceled/Margin/Rejected
2021-01-11, Buy SH600588 Order Canceled/Margin/Rejected
2021-01-11, Buy SH600963 Order Canceled/Margin/R